# 第 10 周 · 周五：案例——用户流失预警设计

> 难度：⭐⭐⭐⭐ | 定位：综合RFM+同期群+行为信号，设计可落地的流失预警方案
> 前置依赖：第2周RFM分群、第4周用户分层、第6周同期群分析、第7周漏斗分析
> 面试地位：用户运营和增长分析岗位的核心考题，考察"从数据到行动"的完整闭环
> 配对课程：01python_learn W10D5（数据分析软技能）

---

## 📘 区块1：核心概念与第一性原理

### 1.1 流失预警的本质

流失预警不是"预测谁会流失"，而是"在用户还没走的时候，告诉运营谁快走了，给运营留出干预时间"。

这个区别非常重要——因为"精准预测14天后会流失的用户"没有意义——14天前他们可能已经没有活跃了，你预测了也没时间干预。真正有价值的是"在用户活跃还在但开始下滑时，第2-3天就发现信号"。

### 1.2 流失预警的四个关键节点

| 节点 | 时间窗口 | 用户状态 | 预警价值 |
|------|---------|---------|--------|
| 潜伏期 | 活跃度开始下降但仍在活跃 | "还在来，但来得少了" | ★★★★★ 黄金窗口 |
| 预警期 | 活跃度明显下降，偶尔来 | "基本不来了，偶尔看一眼" | ★★★★ 最后机会 |
| 沉默期 | 连续N天未活跃 | "已经不来了" | ★★ 只能做召回 |
| 流失期 | 超过流失定义窗口 | "可以认定为流失" | ★ 统计意义 |

**最重要的认知**：好的流失预警不是在"流失期"做判断，而是在"潜伏期"就发现信号。活跃频率下降这个信号，比等到14天不活跃后再判断有价值10倍。

### 1.3 从第一性原理推导：什么是"流失信号"？

流失信号是一个能够提前预测流失行为且可被运营干预的指标变化。一个好的流失信号必须满足三个条件：

1. **先行性**：信号出现在流失行为之前（不能"马后炮"）
2. **可干预性**：信号出现后，运营有手段可以干预（"用户心情不好"不是可干预信号）
3. **高信噪比**：信号能有效区分"会流失的用户"和"不会流失的用户"

### 1.4 流失的定义——整个分析的地基

**为什么重要**：如果"流失"的定义不对，后面的所有分析都是空中楼阁。

**不同产品的流失定义**：

| 产品类型 | 典型流失周期 | 为什么 |
|---------|------------|------|
| 社交/通讯App | 7-14天 | 高频刚需，7天不打开大概率不用了 |
| 电商/购物App | 30-60天 | 中频需求，30天不浏览算信号 |
| 工具类App | 14-30天 | 看工具性质 |
| 游戏 | 3-7天 | 高频娱乐，3天不登录已经是强信号 |
| SaaS/企业服务 | 30-90天 | 低频但持续，月活用户30天为流失 |
| 内容/视频App | 7-14天 | 高频消费，7天不看内容习惯在变 |

**定义原则**：按自然使用周期来定义，而非拍脑袋。方法——画"距上次活跃天数的分布"，找到"正常用户"和"异常用户"的分界点。

### 1.5 面试高频考点

**面试官常问**：
- "如何定义流失？不同产品怎么定？"
- "你设计一个流失预警系统，从0到1怎么做？"
- "什么信号最能预测流失？"
- "预测模型 vs 规则系统，你选哪个？为什么？"

**回答要点**：
- 流失定义要基于产品的自然使用周期
- 0到1：定义流失→找信号→建规则→自动触发→效果验证→持续优化
- 最强的信号是"活跃频率的下降趋势"和"核心功能使用频率下降"
- 规则系统优先——因为可解释、可干预、上线快；模型作为补充

## 🧩 区块2：知识点详细拆解

### 2.1 流失信号的六大维度

| 信号维度 | 指标示例 | 先行性 | 可干预性 | 信噪比 |
|---------|---------|--------|---------|--------|
| 活跃频率 | 过去7天活跃天数/过去30天活跃天数 | ★★★★★ | ★★★★ | ★★★★ |
| 活跃趋势 | (本周活跃-上周活跃)/上周活跃 | ★★★★★ | ★★★ | ★★★★ |
| 核心行为 | 搜索次数、发布次数、支付次数 | ★★★★ | ★★★★★ | ★★★★★ |
| 使用深度 | 浏览页面数、使用时长、访问功能数 | ★★★★ | ★★★ | ★★★ |
| 体验质量 | Crash率、加载失败率、客服投诉 | ★★★ | ★★★★★ | ★★★ |
| 外部因素 | 竞品下载量、宏观环境、季节 | ★★ | ★ | ★ |

### 2.2 RFM在流失预警中的应用

**为什么重要**：RFM（Recency/Frequency/Monetary）是最经典的客户价值分层框架，它同时也能用于流失预警。

| RFM维度 | 与流失的关系 | 预警用途 |
|---------|------------|--------|
| R（最近一次活跃） | 直接相关——R越大流失风险越高 | 核心预警指标 |
| F（活跃频率） | 强相关——F越低的用户越容易流失 | 识别"低频习惯"用户 |
| M（金额/价值） | 间接相关——高M用户的流失容忍度更高 | 区分"值得救"和"不值得救" |

**RFM流失预警矩阵**：
```
                    F(频率)高              F(频率)低
R(最近)小 + M高    ★ 核心用户             ★★ 重要用户
                   关注，不要打扰         偶尔提醒，保持关系

R(最近)大 + M高    ★★★ 休眠高价值         ★★★★ 高流失风险
                   尽快干预，投入资源      最高优先级，立即联系

R(最近)小 + M低    ★★ 活跃低价值          ★ 流失低价值
                   引导升级消费            低成本自动化挽回
```

### 2.3 同期群流失曲线——看透"流失规律"

**为什么重要**：单看某一天的流失率没有意义，看同期群流失曲线才能发现规律。

**同期群分析可以提供的信息**：
1. **哪个同期群的流失更快？** —— 新用户留存是否在改善？
2. **流失是否集中在某个时间窗口？** —— 第3天流失最多？第7天？
3. **不同渠道的新用户流失曲线一样吗？** —— 哪个渠道的用户质量最高？
4. **产品改版后流失曲线有变化吗？** —— 新版本是否留住了更多用户？

**对预警的启示**：如果在同期群分析中发现"第7天是流失高峰期"，那么预警系统就应该在第5-6天触发干预，而不是等到第14天。

### 2.4 漏斗分析——在哪一步流失的？

**为什么重要**：知道"用户在流失"不够，还需要知道"用户在哪个环节流失"。

**常见流失漏斗**：
```
下载App → 注册 → 首次核心体验 → 次日回访 → 第7天回访 → 养成习惯
100%  →  70%  →     45%      →    30%    →    20%     →   15%
         ↓30%       ↓25%           ↓15%        ↓10%        ↓5%
      注册流失     体验流失       次日流失     7日流失      习惯未养成
```

**不同流失阶段的干预策略**：
- 注册流失 → 简化注册流程、第三方登录
- 体验流失 → 优化新手引导、Aha Moment前置
- 次日流失 → Push提醒、新人福利
- 7日流失 → 内容推荐、社交功能引导
- 习惯未养成 → 签到/打卡机制、会员权益

### 2.5 预警系统设计——规则系统 vs 预测模型

**为什么重要**：这是预警系统设计的核心决策。

| 维度 | 规则系统 | 预测模型(ML) |
|------|---------|------------|
| 可解释性 | ★★★★★ 每一条规则都透明 | ★★ 黑盒/灰盒 |
| 可干预性 | ★★★★★ 规则告诉你"该做什么" | ★★★ 模型告诉你"会流失"但不说"为什么" |
| 上线速度 | ★★★★★ 2天可上线 | ★★ 需要2-4周 |
| 维护成本 | ★★★ 新业务需要新规则 | ★★★★ 模型自动学习新pattern |
| 精准度 | ★★★ 可能漏掉非典型pattern | ★★★★★ 理论上可捕捉任何pattern |
| 冷启动 | ★★★★ 业务经验可直接转化为规则 | ★★ 需要足够训练数据 |

**推荐方案**：规则系统作为主体（确保可干预性），ML模型作为补充（捕捉复杂pattern）。

**三灯预警系统设计**：
```
🔴 红灯（立即干预）：
  - 过去7天活跃天数 < 2天
  - 且趋势为"下降"（本周<上周）
  - 且距上次活跃 > 5天
  - 且用户价值 > 中位值（值得救）

🟡 黄灯（关注）：
  - 过去7天活跃天数 2-4天
  - 且趋势为"下降"或"首次出现下降"

🟢 绿灯（正常）：
  - 活跃天数 >= 5天
  - 或趋势为"上升"
```

### 2.6 干预效果评估——如何知道你的预警"救了"多少人？

**为什么重要**：没有效果评估的预警系统是没有价值的——你不知道是在"救人"还是在"骚扰"正常用户。

**评估方法**：

1. **随机化实验**（黄金标准）：对预警触发的用户随机分两组——发送干预 vs 不发送。比较两组的7天留存率差异。

2. **PSM匹配法**（无实验时）：对"被干预的用户"和"特征相似但未被干预的用户"匹配，比较留存差异。

3. **增益分析**：
```
干预增益 = 干预组7天留存率 - 对照组7天留存率
干预ROI = (干预增益 × 用户价值) / 干预成本
```

### 2.7 流失预警的常见陷阱

**陷阱1：过度预警**——阈值太敏感导致大量正常用户被误判为"风险用户"
- 表现：预警用户中90%其实不会流失（精确率=10%）
- 后果：运营团队疲于奔命，用户被过度骚扰

**陷阱2：滞后预警**——信号太晚导致没有干预窗口
- 表现：预警触发时用户已经7天没来了
- 后果：干预效果极差（召回率很高但都是"已死"的用户）

**陷阱3：幸存者偏差**——只看"被预警后没流失"的用户，忽视"被预警后还是流失了"的用户
- 表现：过度乐观估计预警效果
- 后果：预警系统被认为"效果好"但实际上干预手段无效

**陷阱4：选择性干预**——只干预高价值用户，低价值用户被系统性放弃
- 这可能是理性的（ROI考虑），但需要明确说明——"我们选择性地拯救了高价值用户"而不是"我们降低了整体流失率"

### 2.8 从预警到自动化——搭建完整的流失管理体系

**为什么重要**：预警系统只是第一步，完整的流失管理体系需要包括"预警→干预→效果追踪→策略优化"的闭环。

**系统架构**：
```
数据采集 → 特征工程 → 规则引擎 → 自动触发 → 干预执行 → 效果回收
   ↑                                                          ↓
   └──────────────── 策略优化（基于效果反馈） ←────────────────┘
```

**各环节要点**：
- 数据采集：T+1就够了（不需要实时，流失预警的窗口在"天"级别）
- 特征工程：活跃天数、距上次活跃、趋势斜率、核心行为次数、Crash次数
- 规则引擎：SQL化（易于维护）+ 阈值可配置（运营可以自己调）
- 自动触发：对接Push/短信/邮件系统 + 内部通知（钉钉/企微/Slack）
- 效果回收：AB实验对比 + 定期review预警规则的精准度和召回率

## 💻 区块3：动手练习

### 步骤0：数据准备——模拟用户行为数据

In [ ]:
import numpy as np; import pandas as pd
import matplotlib.pyplot as plt; from scipy import stats
import warnings; warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
np.random.seed(42)

n_users = 5000
active_days_30d = np.random.poisson(12, n_users)
active_days_30d = np.clip(active_days_30d, 0, 30)
days_since_last = np.random.exponential(5, n_users)
days_since_last = np.clip(days_since_last, 0, 45)

trend_options = ['上升', '平稳', '下降']
active_7d = np.random.poisson(4, n_users)
active_prev_7d = active_7d + np.random.choice([-3, -2, -1, 0, 1, 2, 3], n_users)
active_prev_7d = np.clip(active_prev_7d, 0, 7)

trend = np.where(active_7d > active_prev_7d + 1, '上升',
                 np.where(active_7d < active_prev_7d - 1, '下降', '平稳'))

core_actions = np.random.poisson(8, n_users)
crash_count = np.random.poisson(0.5, n_users)
user_value = np.random.lognormal(5, 0.8, n_users)

churned = days_since_last > 14

df_users = pd.DataFrame({
    'user_id': range(1, n_users + 1),
    'active_days_30d': active_days_30d,
    'active_days_7d': active_7d,
    'active_days_prev_7d': active_prev_7d,
    'days_since_last': days_since_last,
    'trend': trend,
    'core_actions_7d': core_actions,
    'crash_count_7d': crash_count,
    'user_value': np.round(user_value, 2),
    'churned': churned
})

print(f"总用户数: {n_users}")
print(f"流失用户: {churned.sum()} ({churned.mean():.1%})")
print(f"\n趋势分布:\n{df_users['trend'].value_counts()}")
print(f"\n数据预览:\n{df_users.head(10).to_string()}")

### 步骤1：基础练习——流失信号分析

In [ ]:
# 步骤1.1：各趋势组的流失率
for t in ['上升', '平稳', '下降']:
    mask = df_users['trend'] == t
    cr = df_users.loc[mask, 'churned'].mean()
    print(f"趋势={t}: 用户数={mask.sum()}, 流失率={cr:.1%}")

# 步骤1.2：活跃频率与流失率
print(f"\n活跃频率与流失率:")
for days in [0, 1, 2, 3, 4, 5, 6, 7]:
    mask = df_users['active_days_7d'] == days
    if mask.sum() > 0:
        cr = df_users.loc[mask, 'churned'].mean()
        print(f"  7天活跃{days}天: 用户={mask.sum()}, 流失率={cr:.1%}")

# 步骤1.3：组合信号的预测能力
print(f"\n组合信号分析:")
signals = [
    ('趋势下降', df_users['trend'] == '下降'),
    ('趋势下降 + 活跃<3天', (df_users['trend'] == '下降') & (df_users['active_days_7d'] < 3)),
    ('趋势下降 + 活跃<3天 + 距上次>5天', (df_users['trend'] == '下降') & (df_users['active_days_7d'] < 3) & (df_users['days_since_last'] > 5)),
]
for name, mask in signals:
    n_flagged = mask.sum()
    cr = df_users.loc[mask, 'churned'].mean()
    recall = df_users.loc[mask, 'churned'].sum() / df_users['churned'].sum()
    precision = df_users.loc[mask, 'churned'].sum() / n_flagged if n_flagged > 0 else 0
    print(f"  {name}: 标记{n_flagged}人, 流失率={cr:.1%}, 召回率={recall:.1%}, 精确率={precision:.1%}")

### 步骤2：进阶练习——预警规则设计与评估

In [ ]:
# 步骤2.1：三灯预警系统实现
def three_tier_alert_system(df):
    conditions = [
        (df['active_days_7d'] <= 2) & (df['trend'] == '下降') & (df['days_since_last'] > 5) & (df['user_value'] > df['user_value'].median()),
        ((df['active_days_7d'].between(2, 4)) & (df['trend'] != '上升')) | ((df['active_days_7d'] > 4) & (df['trend'] == '下降')),
        (df['active_days_7d'] >= 5) | (df['trend'] == '上升')
    ]
    choices = ['🔴 红灯', '🟡 黄灯', '🟢 绿灯']
    df['alert_level'] = np.select(conditions, choices, default='🟢 绿灯')
    
    for level in ['🔴 红灯', '🟡 黄灯', '🟢 绿灯']:
        mask = df['alert_level'] == level
        cr = df.loc[mask, 'churned'].mean()
        print(f"{level}: {mask.sum()}人 ({mask.sum()/len(df)*100:.1f}%), 流失率={cr:.1%}")
    return df

df_users = three_tier_alert_system(df_users)

In [ ]:
# 步骤2.2：同期群流失曲线
def cohort_churn_curve():
    np.random.seed(42)
    cohorts = ['2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06']
    days = range(1, 31)
    fig, ax = plt.subplots(figsize=(12, 5))
    for i, cohort in enumerate(cohorts):
        base_retention = 0.75 - i * 0.05
        retention = [base_retention * np.exp(-d * 0.06) + 0.15 for d in days]
        retention = [r + np.random.normal(0, 0.01) for r in retention]
        ax.plot(days, retention, linewidth=2, label=cohort, alpha=0.8)
    ax.set_xlabel('注册后天数', fontsize=12)
    ax.set_ylabel('留存率', fontsize=12)
    ax.set_title('同期群留存曲线——流失模式分析', fontsize=14, fontweight='bold')
    ax.legend(title='注册月份')
    ax.grid(True, alpha=0.3)
    ax.axvline(x=7, color='r', linestyle='--', alpha=0.5, label='第7天预警窗口')
    ax.axvline(x=14, color='orange', linestyle='--', alpha=0.5, label='第14天流失定义')
    ax.legend()
    plt.tight_layout(); plt.show()
    print("\n发现: 前7天流失速度最快（曲线最陡）→ 预警应在第3-5天触发")

cohort_churn_curve()

### 步骤3：挑战练习——预警效果模拟

In [ ]:
# 步骤3.1：模拟干预效果
def simulate_intervention_impact(df, intervention_lift=0.20):
    """模拟对红灯用户发送干预后的效果"""
    red_mask = df['alert_level'] == '🔴 红灯'
    n_red = red_mask.sum()
    n_red_churned = df.loc[red_mask, 'churned'].sum()
    churn_rate_before = n_red_churned / n_red
    saved = int(n_red_churned * intervention_lift)
    churn_rate_after = (n_red_churned - saved) / n_red
    print(f"红灯用户: {n_red}人")
    print(f"  干预前流失率: {churn_rate_before:.1%}")
    print(f"  干预后预计流失率: {churn_rate_after:.1%}")
    print(f"  预计挽救: {saved}人")
    avg_value = df.loc[red_mask, 'user_value'].mean()
    print(f"  挽救用户价值: ¥{saved * avg_value:,.0f}")
    return saved

simulate_intervention_impact(df_users)

# 步骤3.2：预警效果评估矩阵
print(f"\n预警效果混淆矩阵:")
for level in ['🔴 红灯', '🟡 黄灯', '🟢 绿灯']:
    mask = df_users['alert_level'] == level
    actual_churn = df_users.loc[mask, 'churned'].sum()
    actual_keep = mask.sum() - actual_churn
    print(f"{level}: 实际流失={actual_churn}, 实际留存={actual_keep}")

In [ ]:
# 步骤3.3：核心行为与流失关系可视化
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
# 图1: 活跃天数 vs 流失率
for i, days in enumerate(range(1, 8)):
    mask = df_users['active_days_7d'] == days
    axes[0].bar(days, df_users.loc[mask, 'churned'].mean() * 100, color='steelblue', alpha=0.7)
axes[0].set_xlabel('7天活跃天数'); axes[0].set_ylabel('流失率(%)')
axes[0].set_title('活跃频率 vs 流失率', fontsize=13)
axes[0].grid(True, alpha=0.3, axis='y')
# 图2: 趋势分布 vs 流失
trend_data = [df_users[df_users['trend'] == t]['churned'].mean() * 100 for t in ['上升', '平稳', '下降']]
axes[1].bar(['上升', '平稳', '下降'], trend_data, color=['green', 'gray', 'red'], alpha=0.7)
axes[1].set_ylabel('流失率(%)'); axes[1].set_title('活跃趋势 vs 流失率', fontsize=13)
axes[1].grid(True, alpha=0.3, axis='y')
# 图3: 核心行为分布
churned_mask = df_users['churned'] == 1
axes[2].hist(df_users.loc[~churned_mask, 'core_actions_7d'], bins=25, alpha=0.5, label='留存用户', color='green')
axes[2].hist(df_users.loc[churned_mask, 'core_actions_7d'], bins=25, alpha=0.5, label='流失用户', color='red')
axes[2].set_xlabel('核心行为次数/7天'); axes[2].set_ylabel('人数')
axes[2].set_title('核心行为分布: 留存 vs 流失', fontsize=13)
axes[2].legend(); axes[2].grid(True, alpha=0.3, axis='y')
plt.suptitle('流失信号多维分析', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

## 📊 区块4：业务场景应用

### 场景1："每个月流失15%的活跃用户"——从诊断到系统

运营总监："我们每个月流失15%的活跃用户。能不能提前知道谁会流失？"

**你的0到1方案**：

**第1周：定义+探索**
- 确定流失定义：距上次活跃>14天
- 探索性分析：什么特征最能区分流失用户和留存用户？
- 发现：活跃频率下降+核心功能使用减少是最强信号

**第2周：规则设计+验证**
- 设计三灯预警规则
- 用历史数据回测：红灯用户中实际流失的概率（精确率）、所有流失用户中被红灯覆盖的比例（召回率）
- 调参平衡：提高阈值→精确率上升但召回率下降

**第3周：上线+实验**
- 红灯用户随机分流：干预组（发Push+短信）vs 对照组（不干预）
- 监控：干预组7天后留存 vs 对照组7天后留存

**第4周：优化+扩展**
- 根据第3周实验效果调整规则
- 扩展：增加细分用户群的差异化干预策略

### 场景2："Push召回"的效果为什么在衰减？

**背景**：你们的Push召回策略效果在持续衰减——半年前召回率12%，现在降到5%。

**诊断**：
1. **用户疲劳**：同一个用户3个月内收到15条Push→第16条的点击率极低
2. **时机不对**：大部分Push在工作日上午发送，但用户活跃高峰在晚上
3. **内容不相关**：推送的通用券，对高价值用户没有吸引力

**优化方案**：
- 频率封顶：每人每周最多2条Push
- 时机优化：根据用户历史活跃时段个性化推送时间
- 内容个性化：高价值用户推VIP权益，低价值用户推新人福利

### 场景3：发现"异常流失"——竞品挖人

**背景**：发现某个用户群的流失率突然从8%飙到22%。下钻发现：这批用户的共同特征是"过去7天没使用核心功能A"。

**分析**：发现这批用户在使用核心功能A时，Crash率高达15%（正常情况下<1%）。根因：竞品在某个机型上投放了大量广告，而这个机型恰好不兼容你们的功能A——用户一用就崩，崩了就卸载，装了竞品。

**对策**：紧急修复兼容性问题 + 对该机型用户发送"我们修复了"的Push + 补偿优惠券。

### 场景4：预测模型 vs 规则系统——谁的"挽留"效果更好？

**场景**：DS团队建了一个XGBoost流失预测模型（AUC=0.82），你设计了规则系统。运营团队应该用哪个？

**你的建议**：
> "模型和规则不冲突——它们解决不同的问题。
> 规则系统适合'可自动化的批量干预'（如Push、短信、优惠券），优势是规则透明、运营可以理解并优化。
> 模型适合'需要精确识别的高价值用户一对一干预'（如VIP用户流失预警），优势是能捕捉复杂模式。
> 建议：规则系统覆盖80%的日常预警，模型标注剩下的20%（高价值+pattern复杂）。"

### 场景5：真实案例——Netflix的"取消订阅"挽留

Netflix不阻止你取消订阅——但你点击"取消"时，他们会展示你看过且评分高的内容预览，再加上"随时可以回来"的文案。这个策略的巧妙之处在于：
- 干预时机精准（在你做决定的瞬间）
- 内容个性化（展示你喜欢的内容）
- 不强迫（尊重用户的决定）

对你的启示：最好的流失预警不是"阻止用户离开"，而是"在用户想离开时提醒他们留下的理由"。

## ✅ 区块5：分级检验清单

### 🟢 初级检验
- [ ] 能说出流失预警的四个关键节点（潜伏/预警/沉默/流失）
- [ ] 能定义至少3种产品的流失周期
- [ ] 能用RFM框架分析流失风险
- [ ] 能设计简单的三灯预警规则

### 🟡 中级检验
- [ ] 能进行同期群流失曲线分析
- [ ] 能计算预警规则的精确率和召回率
- [ ] 能设计干预效果的AB实验方案
- [ ] 能区分预警系统设计中的"过度预警"和"滞后预警"

### 🔴 高级检验
- [ ] 能设计完整的流失管理体系（预警→干预→追踪→优化）
- [ ] 能评估预警的增量价值（挽救了多少"本来会流失但被救回"的用户）
- [ ] 能在规则系统和预测模型之间做出合理的架构选择
- [ ] 能向管理层量化"不做流失预警"的机会成本


## 🔗 区块6：跨模块连接

| 连接方向 | 关联内容 | 为什么有关 |
|---------|---------|----------|
| **← 前置依赖** | 第2周RFM分群 | RFM是流失预警的基础分层框架 |
| **← 前置依赖** | 第6周同期群分析 | 同期群流失曲线揭示流失规律 |
| **← 前置依赖** | 第7周漏斗分析 | 识别用户在哪一步流失 |
| **← 前置依赖** | 第4周用户分层 | 高价值用户流失优先干预 |
| **→ 后续用途** | W11综合项目 | 综合项目中需要设计流失预警方案 |
| **业务连接** | 用户运营CRM | 预警触发后的自动化干预 |
| **业务连接** | 用户增长 | 流失减少=留存提升=DAU增长 |

### 横向连接
**与AB实验的连接**：干预效果评估需要通过AB实验来衡量增量效果
**与机器学习的连接**：XGBoost/CatBoost可以提升预测精准度，但要以可解释性为前提
**与产品分析的连接**：流失根因最终要靠产品改进来解决——预警只是"延缓"，产品才是"根治"

## 📝 区块7：当日小结

> **流失预警不是在"流失期"做判断，而是在"潜伏期"发现信号。活跃频率的下降趋势是预测流失的最强信号——比等到14天不活跃后再判断有价值10倍。好的预警系统有三个核心设计：三灯分层（红灯立即干预/黄灯关注/绿灯正常）、规则可解释（运营能理解为什么被预警）、效果可衡量（AB实验验证预警的增量价值）。记住：预警的最终目标不是"精准预测"，而是"给运营留出干预时间"。**

**五条金句带回家**：
1. "你会在哪天流失"不是好问题——"你现在有什么信号显示你即将流失"才是
2. 活跃频率下降 > 距上次活跃时间 > 用户静态属性——动态行为信号远优于静态标签
3. 预警系统设计的第一原则：规则可解释 > 模型精度
4. 流失预警不验证增量价值 = 不知道是在"救人"还是"骚扰人"
5. 最好的流失管理不是"挽留所有人"，而是"在最值得挽留的人身上花最多精力"

### 🔖 课后拓展
1. **练习**：选一个你常用的App，设计它的流失预警三灯规则
2. **思考题**：如果预警系统的精确率只有15%（85%是误报），是否还有价值？
3. **预习**：明天的案例——如何撰写一份"有人看"的分析报告
4. **面试准备**：回答思路——"如何设计流失预警系统？"、"流失定义怎么定？"

---
*03analyst_thinking · Week 10 Day 5 · 2026-05-21*